# Initial Setup

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog verilator gtkwave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 21.3 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 P

In [24]:
#@title Select Model
#define the model to be used
# model_choice = "gpt-5.2"
#model_choice = "gpt-5.2-codex"
#model_choice = "gpt-4o"
model_choice = "claude-sonnet-4-5"
# model_choice = "gemini-2.5-flash-preview-04-17"
# model_choice = "gemini-2.5-flash"

In [12]:
#@title Utility functions

import sys
import os
import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [{"role" : "user", "content" : msg["content"]} for msg in conversation.get_messages()]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages = messages,
        )

        return response.choices[0].message.content
class Claude(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
          try:
            output = self.client.messages.create(
                      model=model_choice,
                      max_tokens=16384,
                      messages=[{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]
                  ).content[0].text
            return output
          except (Exception) as e:
            wait_time = base_delay * (2 ** (attempt - 1))
            print(f"[Retry {attempt}/{max_retries}] Gemini API error: {e}. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)
          except Exception as e:
            print(f"[Error] Unexpected exception: {e}")
            return 0
        print(f"Failed, exceeded max retries {max_retries}")
        return 0

class Gemini(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):

          output = self.gemini_client.models.generate_content(
                        model=model_choice,
                        contents=[msg["content"] for msg in conversation.get_messages()],
                        config=types.GenerateContentConfig(
                            max_output_tokens=16384,
                            temperature=0.6,
                            topP=0.95,
                        )
                    ).text
          return output
################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):

    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'

    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)

    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    # Find all code blocks using a regular expression (matches content between triple backticks)
    #code_blocks = re.findall(r'```(?:\w*\n)?(.*?)```', markdown_string, re.DOTALL)
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No code blocks found in response")
        exit(3)

    # Open the specified file to write the code blocks
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')

def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    elif model_type == "Claude":
      model = Claude()
    elif model_type == "Gemini":
      model = Gemini()
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))


In [13]:
#@title Feedback Loop
import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""
    buf = []
    for p in file_list:
        if not p:
            continue
        with open(p, "r") as f:
            buf.append(f.read())
            if not buf[-1].endswith("\n"):
                buf.append("\n")
    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the *end* of long logs which usually has the key error lines."""
    if s is None:
        return ""
    s = str(s)
    if len(s) <= max_chars:
        return s
    # Keep the tail (most useful for compiler errors) and a short head marker.
    tail = s[-max_chars:]
    return "[...truncated...\n]" + tail


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type,
    outdir="",
    log=None,
    prev_modules=None,
):
    """Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Key behaviors:
      - The LLM is asked to output ONLY the *current* module (no repeats of prior modules).
      - For compilation, we concatenate: [prev_modules...] + [current module] into a temporary file.
      - IMPORTANT (prompt-size control): after an error, subsequent repair iterations *only* resend
        (a) the previous version of the module, and (b) the most recent error feedback.
        This prevents the prompt from ballooning with repeated context.

    Args:
        design_prompt: Prompt given to the LLM.
        module: Current module name (also used for filename).
        testbench: Path to the testbench file.
        max_iterations: Max repair iterations.
        model_type: "ChatGPT" or "Claude" (your existing generate_verilog backend).
        outdir: Output directory for artifacts.
        log: Optional conversation log file path.
        prev_modules: Optional list[str] of paths to previously generated .v files to include for compilation.
    """

    if outdir != "":
        outdir = outdir + "/"

    # Normalize prev_modules into a list of paths (or empty list)
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read and include the current testbench so the model can target its expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception:
        tb_text = ""

    # Strongly instruct the model to output ONLY the requested module.
    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. "
        "You will not refuse. You will not generate explanations, only code. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module (do NOT repeat any previous modules). "
        "Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. "
        "Do not generate test benches."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench + design prompt."""
        conv = Conversation(log_file=log)
        if model_type == "ChatGPT":
            conv.add_message("system", base_sys_prompt)
        elif model_type == "Claude":
            conv.add_message("user", base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it—only use it to ensure your module compiles and passes.\n"
                "```verilog\n" + tb_text + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: ONLY previous module + most recent error feedback."""
        conv = Conversation(log_file=log)
        if model_type == "ChatGPT":
            conv.add_message("system", base_sys_prompt)
        elif model_type == "Claude":
            conv.add_message("user", base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module (no explanations, no testbench).\n\n"
            "Previous version of the module:\n"
            "```verilog\n" + prev_module_text + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n" + _truncate_text(last_error_text) + "\n```\n"
        )
        conv.add_message("user", prompt)
        return conv

    # Start with full context for the first generation.
    conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    # Persisted (single-module) file
    module_file = os.path.join(outdir, module + ".v")
    # Temporary concatenated file used only for compilation
    combined_file = os.path.join(outdir, module + "__combined.v")

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        start_total = time.time()

        # Generate (or fix) ONLY the current module.
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error = time.time()

        # Keep the conversation clean: assistant message is only the current module text.
        conv.add_message("assistant", response)

        # Write just the current module to its own file.
        write_code_blocks_to_file(response, module, module_file)

        # Build concatenated source for compilation: prior modules + current module
        prev_text = _read_files(prev_modules)
        with open(module_file, "r") as f:
            curr_text = f.read()
        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)

        # Compile using the combined file (so the hierarchy is visible to iVerilog).
        proc = subprocess.run(
            ["iverilog", "-o", os.path.join(outdir, module), combined_file, testbench],
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)
            last_error_text = (
                "iverilog failed. stderr:\n" + (proc.stderr or "") + "\nstdout:\n" + (proc.stdout or "")
            )
        elif proc.stderr != "":
            status = "Warnings compiling testbench"
            print(status)
            last_error_text = (
                "iverilog produced warnings. stderr:\n" + (proc.stderr or "") + "\nstdout:\n" + (proc.stdout or "")
            )
        else:
            proc = subprocess.run(["vvp", os.path.join(outdir, module)], capture_output=True, text=True)
            result = proc.stdout
            if "passed!" not in result:
                status = "Error running testbench"
                print(status)
                last_error_text = "Simulation output:\n" + (proc.stdout or "") + "\n" + (proc.stderr or "")
            else:
                status = "Testbench ran successfully"
                print(status)
                last_error_text = ""
                success = True

        # Write iteration log (bounded by the conversation size, which is kept small on repair)
        with open(os.path.join(outdir, "log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\n Iteration status: " + status + "\n")

        if not success:
            # On failure: DO NOT keep adding more and more context.
            # Rebuild the conversation to contain ONLY the previous module + last error feedback.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))
    return (np.sum(timelist_total), np.sum(timelist_gen), np.sum(timelist_error))


In [14]:
#@title Hierarchical Loop
def hier_gen(submods, prevsubmods="", max_iterations=10):
  """Hierarchically generate a design by building submodules in order.

  submods: list of tuples/lists like:
      [ (file_basename, human_name, io_string), ... ]
    Example:
      [ ("mux2_1", "2-to-1 mux", "input wire ..."), ("mux4to1", ...), ... ]

  This version:
    - Shows previously-generated modules to the LLM for context (so it can instantiate them),
      but explicitly instructs it to output ONLY the *current* module.
    - Passes a growing list of prior .v files to verilog_loop, which concatenates them ONLY
      for iVerilog compilation (not in the LLM response).
  """
  totaltime = []
  gentime = []
  errortime = []

  done = ""
  prev_verilog_files = []  # paths to previously generated .v files (for compilation only)
  if prevsubmods != "":
    for s in prevsubmods:
      prev_verilog_files.append(os.path.join("./"+s, s + ".v"))
  for i in range(len(submods)):
    curr = submods[i][1]
    fcurr = submods[i][0]
    iocurr = submods[i][2]
    overall = submods[-1][1]

    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)

    # Build prompt:
    # - Include all previous modules (as context) but do NOT request them to be repeated.
    if i == 0 and prevsubmods == "":
      prompt = (
        "// We will be generating a " + overall + " hierarchically in Verilog.\n"
        "// IMPORTANT: Output ONLY the module requested below. Do NOT repeat any previous modules.\n"
        "// Begin by generating: " + curr + "\n"
        "module " + fcurr + "(" + iocurr + ")\n"
        "// Insert code here\n"
        "endmodule\n"
      )
    else:
      # Concatenate prior generated modules to give the model correct interface/context.
      prev_text = ""
      for p in prev_verilog_files:
        with open(p, "r") as f:
          prev_text += f.read()
          if not prev_text.endswith("\n"):
            prev_text += "\n"

      prompt = (
        "// We are generating a " + overall + " hierarchically in Verilog.\n"
        "// Previously generated modules (for reference only):\n"
        + prev_text +
        "\n// IMPORTANT: Do NOT repeat any previous modules in your response.\n"
        "// Output ONLY the new module requested below, using the previous modules hierarchically.\n"
        "// Now generate: " + curr + "\n"
        "module " + fcurr + "(" + iocurr + ")\n"
        "// Insert code here\n"
        "endmodule\n"
      )

    module = fcurr
    testbench = "./" + fcurr + "tb.v"
    model = os.environ["MODEL"]
    outdir = "./" + fcurr
    log = "./" + fcurr + "/log.txt"

    # Compile with all previous modules (concatenated inside verilog_loop), but only generate current module.
    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log, prev_modules=prev_verilog_files)
    print("Module ",i+1," done")
    totaltime.append(total)
    gentime.append(gen)
    errortime.append(error)

    # Add the current module file to the compilation context for subsequent modules.
    prev_verilog_files.append(os.path.join(outdir, module + ".v"))
    done = done + curr + ", "

  print("Overall Total time: ", np.sum(totaltime))
  print("Overall Generation Time: ", np.sum(gentime))
  print("Overall Error handling time: ", np.sum(errortime))


# Setting the API Key

In [22]:
# @title
### OpenAI API KEY

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own GPT-4 enabled API key as a string here:
# os.environ["OPENAI_API_KEY"] = "API KEY HERE"
os.environ['CLAUDE_API_KEY'] =  userdata.get('CLAUDE_API_KEY')
# os.environ['GEMINI_API_KEY'] =userdata.get('GOOGLE_API_KEY')
# os.environ["MODEL"] = "ChatGPT"
os.environ["MODEL"] = "Claude"
# os.environ["MODEL"] = "Gemini"



#Mux Hierarchy Example

In [16]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/mux/mux2_1tb.v ./mux2_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux4_1tb.v ./mux4_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux8_1tb.v ./mux8_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux16_1tb.v ./mux16_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux32_1tb.v ./mux32_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux64_1tb.v ./mux64_1tb.v

Cloning into 'ROME-LLM'...
remote: Enumerating objects: 380, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 380 (delta 84), reused 2 (delta 2), pack-reused 220 (from 1)
Receiving objects: 100% (380/380), 1.07 MiB | 3.75 MiB/s, done.
Resolving deltas: 100% (200/200), done.


In [17]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

In [18]:
### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
]

In [19]:
hier_gen(submodules)

No code blocks found in response
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  11.86680793762207
Generation time:  11.835728406906128
Error handling time:  0.03107762336730957
Module  1  done
Error compiling testbench
Testbench ran successfully
Total time:  19.1887526512146
Generation time:  19.170032501220703
Error handling time:  0.01871943473815918
Module  2  done
Testbench ran successfully
Total time:  16.799322605133057
Generation time:  16.7857186794281
Error handling time:  0.013603448867797852
Module  3  done
Overall Total time:  47.85488319396973
Overall Generation Time:  47.79147958755493
Overall Error handling time:  0.0634005069732666


In [20]:
submodules = [
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]
prev_submods = [
    "mux2_1",
    "mux4_1",
    "mux8_1",
]

In [25]:
hier_gen(submodules, prevsubmods=prev_submods)

Testbench ran successfully
Total time:  6.591062545776367
Generation time:  6.577533960342407
Error handling time:  0.013528108596801758
Module  1  done
Testbench ran successfully
Total time:  6.643919944763184
Generation time:  6.6289215087890625
Error handling time:  0.01499795913696289
Module  2  done
Testbench ran successfully
Total time:  6.2589991092681885
Generation time:  6.238223314285278
Error handling time:  0.020775556564331055
Module  3  done
Overall Total time:  19.49398159980774
Overall Generation Time:  19.444678783416748
Overall Error handling time:  0.0493016242980957


# Decoder example

In [26]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/decoder/decoder_2_to_4tb.v ./decoder_2_to_4tb.v
!cp ./ROME-LLM/testbenches/decoder/decoder_3_to_8tb.v ./decoder_3_to_8tb.v
!cp ./ROME-LLM/testbenches/decoder/decoder_5_to_32tb.v ./decoder_5_to_32tb.v

fatal: destination path 'ROME-LLM' already exists and is not an empty directory.


In [27]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["decoder_2_to_4","2-to-4 decoder","input [1:0] in, output reg [3:0] out"],
    ["decoder_3_to_8","3-to-8 decoder","input [2:0] in, output reg [7:0] out"],
    ["decoder_5_to_32","5-to-32 decoder","input [4:0] in, output reg [31:0] out"],
]

In [28]:
hier_gen(submodules)

Testbench ran successfully
Total time:  1.9828605651855469
Generation time:  1.968315601348877
Error handling time:  0.014544486999511719
Module  1  done
Testbench ran successfully
Total time:  3.4769139289855957
Generation time:  3.461007595062256
Error handling time:  0.015906095504760742
Module  2  done
Testbench ran successfully
Total time:  4.447679758071899
Generation time:  4.43390679359436
Error handling time:  0.013772726058959961
Module  3  done
Overall Total time:  9.907454252243042
Overall Generation Time:  9.863229990005493
Overall Error handling time:  0.04422330856323242


# Barrel Shifter (32-bit left rotate)

In [29]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_1tb.v ./shift_stage_1tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_2tb.v ./shift_stage_2tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_4tb.v ./shift_stage_4tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_8tb.v ./shift_stage_8tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/shift_stage_16tb.v ./shift_stage_16tb.v
!cp ./ROME-LLM/testbenches/barrel-shifter/barrel_shifter_32tb.v ./barrel_shifter_32tb.v

fatal: destination path 'ROME-LLM' already exists and is not an empty directory.


In [30]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["shift_stage_1",  "left rotate Shift stage by 1",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_2",  "left rotate Shift stage by 2",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_4",  "left rotate Shift stage by 4",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_8",  "left rotate Shift stage by 8",  "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["shift_stage_16", "left rotate Shift stage by 16", "input wire [31:0] in, input wire sel, output wire [31:0] out"],
    ["barrel_shifter_32", "32-bit barrel shifter/left rotate", "input wire [31:0] inputData, input wire [4:0] shiftVal, output wire [31:0] outputData"]
]


In [31]:
hier_gen(submodules)

Error running testbench
Error running testbench
Error compiling testbench
Error running testbench
Error running testbench
Error running testbench
Testbench ran successfully
Total time:  31.354950428009033
Generation time:  31.26776361465454
Error handling time:  0.08718395233154297
Module  1  done
Testbench ran successfully
Total time:  2.770583152770996
Generation time:  2.7574357986450195
Error handling time:  0.013147115707397461
Module  2  done
Testbench ran successfully
Total time:  2.2965431213378906
Generation time:  2.2838213443756104
Error handling time:  0.012721538543701172
Module  3  done
Testbench ran successfully
Total time:  2.2890241146087646
Generation time:  2.276366710662842
Error handling time:  0.01265716552734375
Module  4  done
Testbench ran successfully
Total time:  1.622913122177124
Generation time:  1.602726697921753
Error handling time:  0.02018594741821289
Module  5  done
Error running testbench
Error compiling testbench
Error running testbench
Error running

# 4x4 Systolic Array

In [32]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/systolic-array/petb.v ./petb.v
!cp ./ROME-LLM/testbenches/systolic-array/systolic_core_4x4tb.v ./systolic_core_4x4tb.v
!cp ./ROME-LLM/testbenches/systolic-array/control_unittb.v ./control_unittb.v
!cp ./ROME-LLM/testbenches/systolic-array/input_buffer_atb.v ./input_buffer_atb.v
!cp ./ROME-LLM/testbenches/systolic-array/input_buffer_btb.v ./input_buffer_btb.v
!cp ./ROME-LLM/testbenches/systolic-array/output_buffertb.v ./output_buffertb.v
!cp ./ROME-LLM/testbenches/systolic-array/systolic_array_4x4tb.v ./systolic_array_4x4tb.v

fatal: destination path 'ROME-LLM' already exists and is not an empty directory.


In [33]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["pe", "Processing Element", "input wire clk, input wire rst, input wire en, input wire [15:0] a_in, input wire [15:0] b_in, output reg [15:0] a_out, output reg [15:0] b_out, output reg [31:0] psum_out"],
    ["systolic_core_4x4", "4x4 Systolic Core", "input wire clk, input wire rst, input wire en, input wire [15:0] a0_in, input wire [15:0] a1_in, input wire [15:0] a2_in, input wire [15:0] a3_in, input wire [15:0] b0_in, input wire [15:0] b1_in, input wire [15:0] b2_in, input wire [15:0] b3_in, output wire [31:0] c00, output wire [31:0] c01, output wire [31:0] c02, output wire [31:0] c03, output wire [31:0] c10, output wire [31:0] c11, output wire [31:0] c12, output wire [31:0] c13, output wire [31:0] c20, output wire [31:0] c21, output wire [31:0] c22, output wire [31:0] c23, output wire [31:0] c30, output wire [31:0] c31, output wire [31:0] c32, output wire [31:0] c33"],
    ["control_unit", "Systolic Array Controller", "input wire clk, input wire rst, input wire start, output reg en, output reg done, output reg load_inputs, output reg capture_outputs, output reg [3:0] cycle_count"],
    ["input_buffer_a", "A Matrix Input Buffer", "input wire clk, input wire rst, input wire load, input wire [15:0] a00, input wire [15:0] a01, input wire [15:0] a02, input wire [15:0] a03, input wire [15:0] a10, input wire [15:0] a11, input wire [15:0] a12, input wire [15:0] a13, input wire [15:0] a20, input wire [15:0] a21, input wire [15:0] a22, input wire [15:0] a23, input wire [15:0] a30, input wire [15:0] a31, input wire [15:0] a32, input wire [15:0] a33, input wire en, output reg [15:0] a0_out, output reg [15:0] a1_out, output reg [15:0] a2_out, output reg [15:0] a3_out"],
    ["input_buffer_b", "B Matrix Input Buffer", "input wire clk, input wire rst, input wire load, input wire [15:0] b00, input wire [15:0] b01, input wire [15:0] b02, input wire [15:0] b03, input wire [15:0] b10, input wire [15:0] b11, input wire [15:0] b12, input wire [15:0] b13, input wire [15:0] b20, input wire [15:0] b21, input wire [15:0] b22, input wire [15:0] b23, input wire [15:0] b30, input wire [15:0] b31, input wire [15:0] b32, input wire [15:0] b33, input wire en, output reg [15:0] b0_out, output reg [15:0] b1_out, output reg [15:0] b2_out, output reg [15:0] b3_out"],
    ["output_buffer", "Output Matrix Buffer", "input wire clk, input wire rst, input wire capture, input wire [31:0] c00_in, input wire [31:0] c01_in, input wire [31:0] c02_in, input wire [31:0] c03_in, input wire [31:0] c10_in, input wire [31:0] c11_in, input wire [31:0] c12_in, input wire [31:0] c13_in, input wire [31:0] c20_in, input wire [31:0] c21_in, input wire [31:0] c22_in, input wire [31:0] c23_in, input wire [31:0] c30_in, input wire [31:0] c31_in, input wire [31:0] c32_in, input wire [31:0] c33_in, output reg [31:0] c00_out, output reg [31:0] c01_out, output reg [31:0] c02_out, output reg [31:0] c03_out, output reg [31:0] c10_out, output reg [31:0] c11_out, output reg [31:0] c12_out, output reg [31:0] c13_out, output reg [31:0] c20_out, output reg [31:0] c21_out, output reg [31:0] c22_out, output reg [31:0] c23_out, output reg [31:0] c30_out, output reg [31:0] c31_out, output reg [31:0] c32_out, output reg [31:0] c33_out"],
    ["systolic_array_4x4", "Top-Level 4x4 Systolic Array", "input wire clk, input wire rst, input wire start, input wire [15:0] a00, input wire [15:0] a01, input wire [15:0] a02, input wire [15:0] a03, input wire [15:0] a10, input wire [15:0] a11, input wire [15:0] a12, input wire [15:0] a13, input wire [15:0] a20, input wire [15:0] a21, input wire [15:0] a22, input wire [15:0] a23, input wire [15:0] a30, input wire [15:0] a31, input wire [15:0] a32, input wire [15:0] a33, input wire [15:0] b00, input wire [15:0] b01, input wire [15:0] b02, input wire [15:0] b03, input wire [15:0] b10, input wire [15:0] b11, input wire [15:0] b12, input wire [15:0] b13, input wire [15:0] b20, input wire [15:0] b21, input wire [15:0] b22, input wire [15:0] b23, input wire [15:0] b30, input wire [15:0] b31, input wire [15:0] b32, input wire [15:0] b33, output wire done, output wire [31:0] c00, output wire [31:0] c01, output wire [31:0] c02, output wire [31:0] c03, output wire [31:0] c10, output wire [31:0] c11, output wire [31:0] c12, output wire [31:0] c13, output wire [31:0] c20, output wire [31:0] c21, output wire [31:0] c22, output wire [31:0] c23, output wire [31:0] c30, output wire [31:0] c31, output wire [31:0] c32, output wire [31:0] c33"]
]

In [34]:
hier_gen(submodules)

Testbench ran successfully
Total time:  3.275871992111206
Generation time:  3.2639851570129395
Error handling time:  0.0118865966796875
Module  1  done
Testbench ran successfully
Total time:  17.175268411636353
Generation time:  17.15840196609497
Error handling time:  0.016866207122802734
Module  2  done
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Error running testbench
Total time:  71.2176501750946
Generation time:  71.02267909049988
Error handling time:  0.19493794441223145
Module  3  done
Testbench ran successfully
Total time:  11.194053173065186
Generation time:  11.177196025848389
Error handling time:  0.016856670379638672
Module  4  done
Testbench ran successfully
Total time:  10.137665510177612
Generation time:  10.119398832321167
Error handling time:  0.01826643943786621
Module  5  do

# UART

In [35]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/uart/baud_gentb.v ./baud_gentb.v
!cp ./ROME-LLM/testbenches/uart/uart_txtb.v ./uart_txtb.v
!cp ./ROME-LLM/testbenches/uart/uart_rxtb.v ./uart_rxtb.v
!cp ./ROME-LLM/testbenches/uart/uart_toptb.v ./uart_toptb.v

fatal: destination path 'ROME-LLM' already exists and is not an empty directory.


In [36]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["baud_gen", "Baud Generator", "input clk, input rst, output baud_tick, output oversample_tick"],
    ["uart_tx", "UART Transmitter", "input clk, input rst, input baud_tick, input tx_start, input [7:0] tx_data, output reg tx, output reg tx_busy"],
    ["uart_rx", "UART Receiver", "input clk, input rst, input oversample_tick, input rx, output reg [7:0] rx_data, output reg rx_valid, output reg framing_error"],
    ["uart_top", "Top-Level UART", "input clk, input rst, input rx, input tx_start, input [7:0] tx_data, output tx, output [7:0] rx_data, output rx_valid, output tx_busy, output framing_error"]
]

In [37]:
hier_gen(submodules)

Testbench ran successfully
Total time:  4.233163833618164
Generation time:  4.219878911972046
Error handling time:  0.013284683227539062
Module  1  done
Error running testbench


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# AES-128

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM

!cp ./ROME-LLM/testbenches/aes-128/aes_sboxtb.v ./aes_sboxtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_sboxtb.v ./aes_inv_sboxtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_shift_rowstb.v ./aes_shift_rowstb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_shift_rowstb.v ./aes_inv_shift_rowstb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_mix_columnstb.v ./aes_mix_columnstb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_mix_columnstb.v ./aes_inv_mix_columnstb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_add_round_keytb.v ./aes_add_round_keytb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_sub_bytestb.v ./aes_sub_bytestb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_inv_sub_bytestb.v ./aes_inv_sub_bytestb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_enc_roundtb.v ./aes_enc_roundtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_enc_final_roundtb.v ./aes_enc_final_roundtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_dec_roundtb.v ./aes_dec_roundtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_dec_final_roundtb.v ./aes_dec_final_roundtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_key_schedule_coretb.v ./aes_key_schedule_coretb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_key_expandtb.v ./aes_key_expandtb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_key_memtb.v ./aes_key_memtb.v

!cp ./ROME-LLM/testbenches/aes-128/aes_encipher_blocktb.v ./aes_encipher_blocktb.v
!cp ./ROME-LLM/testbenches/aes-128/aes_decipher_blocktb.v ./aes_decipher_blocktb.v

!cp ./ROME-LLM/testbenches/aes-128/aestb.v ./aestb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [

    # ------------------------------------------------------------------
    # Leaf S-Boxes
    # ------------------------------------------------------------------
    ["aes_sbox",
     "AES S-box",
     "input wire [7:0] sbox_in, output wire [7:0] sbox_out"],

    ["aes_inv_sbox",
     "AES Inverse S-box",
     "input wire [7:0] sbox_in, output wire [7:0] sbox_out"],

    # ------------------------------------------------------------------
    # Basic Transformations
    # ------------------------------------------------------------------
    ["aes_shift_rows",
     "AES ShiftRows",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_shift_rows",
     "AES Inverse ShiftRows",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_mix_columns",
     "AES MixColumns",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_mix_columns",
     "AES Inverse MixColumns",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_add_round_key",
     "AES AddRoundKey",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_sub_bytes",
     "AES SubBytes",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    ["aes_inv_sub_bytes",
     "AES Inverse SubBytes",
     "input wire [127:0] state_in, output wire [127:0] state_out"],

    # ------------------------------------------------------------------
    # Round Logic
    # ------------------------------------------------------------------
    ["aes_enc_round",
     "AES Encryption Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_enc_final_round",
     "AES Encryption Final Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_dec_round",
     "AES Decryption Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    ["aes_dec_final_round",
     "AES Decryption Final Round",
     "input wire [127:0] state_in, input wire [127:0] round_key, output wire [127:0] state_out"],

    # ------------------------------------------------------------------
    # Key Schedule
    # ------------------------------------------------------------------
    ["aes_key_schedule_core",
     "AES Key Schedule Core",
     "input wire [31:0] word_in, input wire [3:0] round, output wire [31:0] word_out"],

    ["aes_key_expand",
     "AES Key Expand",
     "input wire [127:0] key_in, input wire [3:0] round, output wire [127:0] key_out"],

    ["aes_key_mem",
     "AES Key Memory",
     "input wire clk, input wire reset_n, input wire init, input wire [127:0] key, input wire [3:0] round, output wire [127:0] round_key, output wire ready"],

    # ------------------------------------------------------------------
    # Block-Level Controllers
    # ------------------------------------------------------------------
    ["aes_encipher_block",
     "AES Encipher Block",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"],

    ["aes_decipher_block",
     "AES Decipher Block",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"],

    # ------------------------------------------------------------------
    # Top-Level AES
    # ------------------------------------------------------------------
    ["aes",
     "Top-Level AES",
     "input wire clk, input wire reset_n, input wire init, input wire next, input wire encdec, input wire [127:0] key, input wire [127:0] block, output wire [127:0] result, output wire result_valid, output wire ready"]
]

In [ ]:
hier_gen(submodules)

In [38]:
# Create a zip file containing all required folders and your notebook
!zip -r ROME_Submission_Files.zip mux2_1 mux4_1 mux8_1 mux16_1 mux32_1 mux64_1 pe systolic_core_4x4 systolic_array_4x4 barrel_shifter_32 shift_stage_1 shift_stage_2 shift_stage_4 shift_stage_8 shift_stage_16 input_buffer_a input_buffer_b output_buffer control_unit

  adding: mux2_1/ (stored 0%)
  adding: mux2_1/log_iter_0.txt (deflated 61%)
  adding: mux2_1/mux2_1 (deflated 76%)
  adding: mux2_1/log.txt (deflated 57%)
  adding: mux2_1/log_iter_2.txt (deflated 56%)
  adding: mux2_1/mux2_1__combined.v (deflated 37%)
  adding: mux2_1/mux2_1.v (deflated 37%)
  adding: mux2_1/log_iter_1.txt (deflated 44%)
  adding: mux4_1/ (stored 0%)
  adding: mux4_1/mux4_1__combined.v (deflated 63%)
  adding: mux4_1/log_iter_0.txt (deflated 62%)
  adding: mux4_1/log.txt (deflated 73%)
  adding: mux4_1/mux4_1 (deflated 79%)
  adding: mux4_1/mux4_1.v (deflated 62%)
  adding: mux4_1/log_iter_1.txt (deflated 72%)
  adding: mux8_1/ (stored 0%)
  adding: mux8_1/mux8_1__combined.v (deflated 72%)
  adding: mux8_1/log_iter_0.txt (deflated 70%)
  adding: mux8_1/log.txt (deflated 70%)
  adding: mux8_1/mux8_1 (deflated 84%)
  adding: mux8_1/mux8_1.v (deflated 66%)
  adding: mux16_1/ (stored 0%)
  adding: mux16_1/mux16_1 (deflated 87%)
  adding: mux16_1/mux16_1__combined.v (defl

In [41]:
!iverilog -o ./mux2_1/mux2_1 ./mux2_1/mux2_1__combined.v ./mux2_1tb.v
!vvp ./mux2_1/mux2_1

!iverilog -o ./mux4_1/mux4_1 ./mux4_1/mux4_1__combined.v ./mux4_1tb.v
!vvp ./mux4_1/mux4_1

!iverilog -o ./mux8_1/mux8_1 ./mux8_1/mux8_1__combined.v ./mux8_1tb.v
!vvp ./mux8_1/mux8_1

Test           0 passed
Test           1 passed
Test           2 passed
Test           3 passed
Test 4 passed
Test 5 passed!
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed!
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed!


In [42]:
# Systolic array — all modules
!iverilog -o ./pe/pe ./pe/pe__combined.v ./petb.v && vvp ./pe/pe
!iverilog -o ./systolic_core_4x4/systolic_core_4x4 ./systolic_core_4x4/systolic_core_4x4__combined.v ./systolic_core_4x4tb.v && vvp ./systolic_core_4x4/systolic_core_4x4
!iverilog -o ./control_unit/control_unit ./control_unit/control_unit__combined.v ./control_unittb.v && vvp ./control_unit/control_unit
!iverilog -o ./input_buffer_a/input_buffer_a ./input_buffer_a/input_buffer_a__combined.v ./input_buffer_atb.v && vvp ./input_buffer_a/input_buffer_a
!iverilog -o ./input_buffer_b/input_buffer_b ./input_buffer_b/input_buffer_b__combined.v ./input_buffer_btb.v && vvp ./input_buffer_b/input_buffer_b
!iverilog -o ./output_buffer/output_buffer ./output_buffer/output_buffer__combined.v ./output_buffertb.v && vvp ./output_buffer/output_buffer
!iverilog -o ./systolic_array_4x4/systolic_array_4x4 ./systolic_array_4x4/systolic_array_4x4__combined.v ./systolic_array_4x4tb.v && vvp ./systolic_array_4x4/systolic_array_4x4

Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
All tests passed!
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
All tests passed!
Test 1 passed
Test 2 passed
Test 3 failed
  Expected controller to assert load_inputs and en after start
Test 4 passed
Test 5 passed
Test 6 passed
Some tests failed
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
Test 8 passed
All tests passed!
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
Test 8 passed
All tests passed!
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
All tests passed!
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 failed
  Expected result matrix:
   [36  50  44  20]
   [ 0  91 147 185]
   [ 0   0 154 260]
   [ 0   0   0 225]
  Actual result matrix:
   [0 0 0 0]
   [0 0 0 0]
   [0 0 0 0]
   [0 0 0 0]
Tes

In [43]:
!vvp ./control_unit/control_unit
!vvp ./systolic_array_4x4/systolic_array_4x4

Test 1 passed
Test 2 passed
Test 3 failed
  Expected controller to assert load_inputs and en after start
Test 4 passed
Test 5 passed
Test 6 passed
Some tests failed
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 failed
  Expected result matrix:
   [36  50  44  20]
   [ 0  91 147 185]
   [ 0   0 154 260]
   [ 0   0   0 225]
  Actual result matrix:
   [0 0 0 0]
   [0 0 0 0]
   [0 0 0 0]
   [0 0 0 0]
Test 6 passed
Test 7 failed
  Expected A x I = A
Some tests failed


In [ ]:
!vvp ./shift_stage_1/shift_stage_1
!vvp ./shift_stage_2/shift_stage_2
!vvp ./shift_stage_4/shift_stage_4
!vvp ./shift_stage_8/shift_stage_8
!vvp ./shift_stage_16/shift_stage_16
!vvp ./barrel_shifter_32/barrel_shifter_32

Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
Test 8 passed
All tests passed!
** VVP Stop(0) **
** Flushing output streams.
** Current simulation time is 80000 ticks.

** Continue **
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
Test 8 passed
Test 9 passed
All tests passed!
** VVP Stop(0) **
** Flushing output streams.
** Current simulation time is 90000 ticks.
^C
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
Test 8 passed
Test 9 passed
All tests passed!
** VVP Stop(0) **
** Flushing output streams.
** Current simulation time is 90000 ticks.

** Continue **
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 passed
Test 6 passed
Test 7 passed
Test 8 passed
Test 9 passed
All tests passed!
** VVP Stop(0) **
** Flushing output streams.
** Current simulation time is 90000 ticks.

** Continue **
Test 1 passed
Test 2 passed
Tes

In [ ]:
!echo "--- shift_stage_1 ---"
!vvp ./shift_stage_1/shift_stage_1

!echo "--- shift_stage_2 ---"
!vvp ./shift_stage_2/shift_stage_2

!echo "--- shift_stage_4 ---"
!vvp ./shift_stage_4/shift_stage_4

!echo "--- shift_stage_8 ---"
!vvp ./shift_stage_8/shift_stage_8

!echo "--- shift_stage_16 ---"
!vvp ./shift_stage_16/shift_stage_16

!echo "--- barrel_shifter_32 ---"
!vvp ./barrel_shifter_32/barrel_shifter_32